# Human MOF Quest benchmark

Analyze the recorded 22-question panel using anonymous participant responses. Reaction IDs align answers with the question key. The workflow runs locally.

## Inputs and setup

Install `python -m pip install -e ".[evaluation,notebook]"` from the repository root. Both anonymous input tables are included. Run the cells in order; the final cell saves local results. No API key or original workbook is required. All paths below are repository-relative.

| Input | Location | Required fields |
| --- | --- | --- |
| Human question key, CSV | `benchmarks/mof_quest/human_questions.csv` | `question`, `reaction_id`, `label`, `difficulty`, `doi`, `conditions_json` |
| Participant responses, CSV | `benchmarks/mof_quest/human_responses.csv` | `participant_id`, `experience`, `reaction_id`, `response`, `stored_score` |

For another cohort, create anonymous CSVs with the same schemas and select them with `QUESTIONS` and `RESPONSES` below. Keep reaction IDs aligned across both tables and preserve response-category spelling. Select a new `OUTPUT_DIR` to save a separate analysis. The included cohort writes to `results/evaluation/human_quest/`.

Implementation: [human benchmark calculations](../src/mofinder/evaluation/human_quest.py). See the [source-to-code guide](../docs/source_to_code.md) for the original workflow stages and their corresponding functions.


In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd().resolve()
while not (ROOT / "pyproject.toml").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
if not (ROOT / "src" / "mofinder").exists():
    raise RuntimeError("Open this notebook from the MOFinder repository.")
sys.path.insert(0, str(ROOT / "src"))

import pandas as pd
from mofinder.evaluation.human_quest import load_benchmark, write_analysis

QUESTIONS = ROOT / "benchmarks/mof_quest/human_questions.csv"
RESPONSES = ROOT / "benchmarks/mof_quest/human_responses.csv"
OUTPUT_DIR = ROOT / "results/evaluation/human_quest"


## Accuracy and agreement

Success responses predict P and failure responses predict N, with the two confidence levels retained separately. Overall uncertainty uses a t interval across participants on this fixed panel.


In [ ]:
result = load_benchmark(QUESTIONS, RESPONSES)
display(pd.Series(result["summary"], name="value").to_frame())


## Experience and confidence

Confidence remains categorical. Response shares use all answers within each experience group as the denominator.


In [ ]:
display(pd.DataFrame(result["experience"]))
display(pd.DataFrame(result["confidence"]))


## Per-question accuracy

Error bars are 95% Wilson intervals. Reference-positive questions are teal and reference-negative questions are grey.


In [ ]:
import matplotlib.pyplot as plt

items = pd.DataFrame(result["questions"])
fig, ax = plt.subplots(figsize=(10, 4))
colors = ["#285953" if label == "P" else "#91999B" for label in items["label"]]
ax.bar(items["question"], items["accuracy"], color=colors)
ax.errorbar(
    items["question"], items["accuracy"],
    yerr=[items["accuracy"] - items["wilson_lower"],
          items["wilson_upper"] - items["accuracy"]],
    fmt="none", ecolor="#333333", capsize=2, linewidth=1,
)
ax.set(xlabel="Question", ylabel="Accuracy", ylim=(0, 1))
ax.tick_params(axis="x", rotation=45)
fig.tight_layout()
plt.show()


## Save results

Save the participant, question, experience, and confidence tables.


In [ ]:
write_analysis(result, OUTPUT_DIR)
print(f"Results saved to {OUTPUT_DIR}")
